# 🩺 Medical Agent — Full Pipeline

Loads the two saved adapters and runs the complete agent with a Gradio UI.

```
Patient complaint
  ↓
  [follow_up_adapter]   → ask focused follow-up questions (max 6 turns)
  ↓
  [base LLM, no adapter] → extract symptom summary for RAG search
  ↓
  [PubMed + MedlinePlus] → retrieve relevant medical literature
  ↓
  [diagnostic_adapter]  → generate structured differential diagnosis
  ↓
  [Gradio UI]           → patient-facing chat interface
```

**Prerequisites — run these first:**
- `finetune_followup.ipynb` → saves `follow_up_adapter/` to Drive
- `finetune_diagnostic.ipynb` → saves `diagnostic_adapter/` to Drive


## 📦 Step 1: Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'unsloth[colab-new]', 'peft', 'accelerate',
    'bitsandbytes', 'gradio>=4.0', 'httpx', '-q'
], check=True)
print('✅ Installed.')

## 💾 Step 2: Mount Drive & Configuration

In [ ]:
from google.colab import drive
import os, gc, re, json, uuid, time
from datetime import datetime
from enum import Enum
import torch

drive.mount('/content/drive', force_remount=False)

# ─── Adapter paths (must match where the training notebooks saved them) ────────
DRIVE_ROOT              = '/content/drive/MyDrive/unsloth_training'
FOLLOW_UP_ADAPTER_PATH  = os.path.join(DRIVE_ROOT, 'follow_up_adapter')
DIAGNOSTIC_ADAPTER_PATH = os.path.join(DRIVE_ROOT, 'diagnostic_adapter')
SESSIONS_DIR            = os.path.join(DRIVE_ROOT, 'sessions')
os.makedirs(SESSIONS_DIR, exist_ok=True)

# ─── Model ────────────────────────────────────────────────────────────────────
BASE_MODEL     = 'unsloth/llama-2-7b-bnb-4bit'
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT   = True

# ─── Conversation ─────────────────────────────────────────────────────────────
MAX_FOLLOWUP_TURNS = 6

# ─── Generation ───────────────────────────────────────────────────────────────
FOLLOWUP_MAX_TOKENS   = 80
EXTRACTION_MAX_TOKENS = 80
DIAGNOSIS_MAX_TOKENS  = 512   # kept under 2048 budget
TEMPERATURE           = 0.7
TOP_P                 = 0.9

# ─── RAG ──────────────────────────────────────────────────────────────────────
PUBMED_MAX_RESULTS      = 3
MEDLINEPLUS_MAX_RESULTS = 2
RAG_MAX_CHARS_PER_DOC   = 400   # ≈100 tokens per doc
MAX_CONV_TOKENS         = 700   # conversation truncation budget

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('✅ Configuration loaded.')
print(f'   Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU    : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'   follow_up_adapter  exists: {os.path.exists(FOLLOW_UP_ADAPTER_PATH)}')
print(f'   diagnostic_adapter exists: {os.path.exists(DIAGNOSTIC_ADAPTER_PATH)}')

## 🗂️ Step 3: Conversation State Machine

In [ ]:
class State(Enum):
    INTAKE     = 'INTAKE'
    FOLLOW_UP  = 'FOLLOW_UP'
    EXTRACTING = 'EXTRACTING'
    RAG        = 'RAG'
    DIAGNOSING = 'DIAGNOSING'
    DONE       = 'DONE'


class Session:
    """
    Tracks one patient session: history, state, extracted symptoms,
    RAG results, and final diagnosis. Saves to Drive as JSON.
    """
    def __init__(self, max_turns: int = MAX_FOLLOWUP_TURNS):
        self.id          = str(uuid.uuid4())[:8]
        self.state       = State.INTAKE
        self.complaint   = ''
        self.history     = []   # [{'role': str, 'text': str}]
        self.turns       = 0
        self.max_turns   = max_turns
        self.symptoms    = ''
        self.rag_docs    = []
        self.diagnosis   = ''

    def patient_says(self, text: str):
        text = text.strip()
        self.history.append({'role': 'Patient', 'text': text})
        if self.state == State.INTAKE:
            self.complaint = text
            self.state     = State.FOLLOW_UP

    def doctor_says(self, text: str):
        self.history.append({'role': 'Doctor', 'text': text.strip()})
        self.turns += 1

    def exhausted(self) -> bool:
        return self.turns >= self.max_turns

    def transcript(self) -> str:
        return '\n'.join(f"{h['role']}: {h['text']}" for h in self.history)

    def history_tuples(self):
        return [(h['role'], h['text']) for h in self.history]

    def save(self) -> str:
        path = os.path.join(SESSIONS_DIR, f'session_{self.id}.json')
        with open(path, 'w') as f:
            json.dump({
                'id'        : self.id,
                'saved_at'  : datetime.now().isoformat(),
                'complaint' : self.complaint,
                'history'   : self.history,
                'turns'     : self.turns,
                'symptoms'  : self.symptoms,
                'rag_sources': [d.get('source','') for d in self.rag_docs],
                'diagnosis' : self.diagnosis,
            }, f, indent=2)
        print(f'💾 Session saved → {path}')
        return path

print('✅ Session state machine defined.')

## 🔬 Step 4: RAG — PubMed + MedlinePlus

In [ ]:
import httpx

class MedicalRAG:
    """
    Free medical literature retrieval.
    PubMed : NCBI E-utilities (no API key, 3 req/sec limit)
    MedlinePlus: NLM Connect API (no API key)
    """
    PUBMED_SEARCH = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi'
    PUBMED_FETCH  = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'
    MEDLINEPLUS   = 'https://connect.medlineplus.gov/application'

    def __init__(self):
        self._c = httpx.Client(timeout=15.0)

    def _pubmed_ids(self, query, n):
        try:
            r = self._c.get(self.PUBMED_SEARCH,
                params={'db':'pubmed','term':query,'retmax':n,'retmode':'json'})
            return r.json()['esearchresult']['idlist']
        except: return []

    def _pubmed_abstracts(self, ids):
        if not ids: return []
        try:
            r = self._c.get(self.PUBMED_FETCH,
                params={'db':'pubmed','id':','.join(ids),'rettype':'abstract','retmode':'text'})
            raw = r.text
        except: return []
        return [{'source':f'PubMed:{pid}',
                 'text'  : raw[:RAG_MAX_CHARS_PER_DOC].strip(),
                 'url'   : f'https://pubmed.ncbi.nlm.nih.gov/{pid}/'}
                for pid in ids]

    def _medlineplus(self, query, n):
        try:
            r = self._c.get(self.MEDLINEPLUS,
                params={'mainSearchCriteria.v.cs':'2.16.840.1.113883.6.90',
                        'mainSearchCriteria.v.dn':query,
                        'knowledgeResponseType':'application/json',
                        'informationRecipient.languageCode.c':'en'})
            data = r.json()
            docs = []
            for e in data.get('feed',{}).get('entry',[])[:n]:
                title   = e.get('title',{}).get('_value','')
                summary = e.get('summary',{}).get('_value','')
                link    = (e.get('link') or [{}])[0].get('href','')
                docs.append({'source':'MedlinePlus',
                             'text'  : f'{title}. {summary}'[:RAG_MAX_CHARS_PER_DOC],
                             'url'   : link})
            return docs
        except:
            # Fallback to plain NLM search
            try:
                r = self._c.get('https://wsearch.nlm.nih.gov/ws/query',
                    params={'db':'healthTopics','term':query,'retmax':n,'rettype':'brief'})
                return [{'source':'MedlinePlus','text':r.text[:RAG_MAX_CHARS_PER_DOC],'url':''}]
            except: return []

    def retrieve(self, query: str) -> list:
        print(f'Searching: "{query[:60]}"')
        ids  = self._pubmed_ids(query, PUBMED_MAX_RESULTS)
        time.sleep(0.35)                              # NCBI rate limit
        docs = self._pubmed_abstracts(ids)
        docs += self._medlineplus(query, MEDLINEPLUS_MAX_RESULTS)
        print(f'Retrieved {len(docs)} document(s)')
        return docs

    @staticmethod
    def format(docs: list) -> str:
        if not docs:
            return 'No relevant literature retrieved.'
        return '\n'.join(f'[{i}] {d["source"]}: {d["text"].strip()}'
                         for i, d in enumerate(docs, 1))

rag = MedicalRAG()
print('MedicalRAG ready.')

## 🤖 Step 5: Load Model + Both Adapters

Both adapters are pre-loaded as **named PEFT adapters** so switching roles
is just `model.set_adapter(name)` — no disk I/O at runtime.
`for_inference()` is called **once** after loading.


In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel

ROLE_FOLLOWUP = 'follow_up'
ROLE_EXTRACT  = 'extract'    # no adapter — raw base model
ROLE_DIAGNOSE = 'diagnose'

_current_role = None

# ── 1. Load base model ────────────────────────────────────────────────────────
print(' Loading base model...')
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = BASE_MODEL,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = None,
    load_in_4bit  = LOAD_IN_4BIT,
)
tokenizer.padding_side = 'left'
print(' Base model loaded.')

# ── 2. Load follow_up_adapter (initialises PEFT wrapper) ─────────────────────
if os.path.exists(FOLLOW_UP_ADAPTER_PATH):
    print(f'Loading follow_up_adapter...')
    model = PeftModel.from_pretrained(
        base_model, FOLLOW_UP_ADAPTER_PATH, adapter_name=ROLE_FOLLOWUP
    )
    print(f'follow_up_adapter loaded as "{ROLE_FOLLOWUP}"')
else:
    print(f'follow_up_adapter not found at {FOLLOW_UP_ADAPTER_PATH}')
    print(' Follow-up role will use base model.')
    model = base_model

# ── 3. Load diagnostic_adapter as second named adapter ────────────────────────
if os.path.exists(DIAGNOSTIC_ADAPTER_PATH):
    print(f' Loading diagnostic_adapter...')
    if hasattr(model, 'load_adapter'):
        model.load_adapter(DIAGNOSTIC_ADAPTER_PATH, adapter_name=ROLE_DIAGNOSE)
    else:
        model = PeftModel.from_pretrained(
            model, DIAGNOSTIC_ADAPTER_PATH, adapter_name=ROLE_DIAGNOSE
        )
    print(f' diagnostic_adapter loaded as "{ROLE_DIAGNOSE}"')
else:
    print(f' diagnostic_adapter not found at {DIAGNOSTIC_ADAPTER_PATH}')
    print(' Diagnosis role will use base model.')

# ── 4. Call for_inference() ONCE ──────────────────────────────────────────────
FastLanguageModel.for_inference(model)
print('\n All adapters ready. for_inference() applied.')

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f' VRAM: {used:.2f} / {total:.1f} GB')

## ⚙️ Step 6: Role Switching & Generation

In [ ]:
def switch_role(role: str):
    """Activates the correct adapter. No disk I/O — adapters are pre-loaded."""
    global _current_role
    if _current_role == role:
        return
    if role == ROLE_EXTRACT:
        if hasattr(model, 'disable_adapters'):
            model.disable_adapters()
    else:
        if hasattr(model, 'enable_adapters'):
            model.enable_adapters()
        if hasattr(model, 'set_adapter'):
            try:
                model.set_adapter(role)
            except Exception as e:
                print(f' set_adapter("{role}") failed: {e}')
    _current_role = role


def truncate_to_tokens(text: str, max_tokens: int) -> str:
    """Keeps the LAST max_tokens of text (most recent context)."""
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= max_tokens:
        return text
    return '[...earlier context truncated...]\n' + \
           tokenizer.decode(ids[-max_tokens:], skip_special_tokens=True)


def generate(prompt: str, max_new_tokens: int) -> str:
    """
    Core generation. Reserves token budget for output.
    for_inference() was already called in Step 5 — do NOT call it here.
    """
    inputs = tokenizer(
        [prompt], return_tensors='pt',
        truncation=True,
        max_length=MAX_SEQ_LENGTH - max_new_tokens,   # leave room for output
    ).to(DEVICE)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = TEMPERATURE,
            top_p              = TOP_P,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
            eos_token_id       = tokenizer.eos_token_id,
            repetition_penalty = 1.1,
        )

    new = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True).strip()


# ── Role-specific generation functions ───────────────────────────────────────

def ask_followup(complaint: str, history: list) -> str:
    """Generates the next follow-up question using follow_up_adapter."""
    switch_role(ROLE_FOLLOWUP)
    hist_str = '\n'.join(f'{r}: {t}' for r, t in history) if history else '(none)'
    prompt = (
        '### Instruction:\n'
        'You are a medical intake doctor. Based on the patient complaint '
        'and conversation history below, ask ONE focused follow-up question. '
        'Do not diagnose.\n\n'
        f'Patient complaint: {complaint}\n'
        f'Conversation so far:\n{hist_str}\n\n'
        '### Response:\n'
    )
    raw = generate(prompt, FOLLOWUP_MAX_TOKENS)
    # Keep only the first question sentence
    m = re.search(r'[^.!\n]*\?', raw)
    return m.group(0).strip() if m else raw.split('\n')[0].strip()


def extract_symptoms(conversation: str) -> str:
    """Uses base model (no adapter) to extract a concise symptom summary."""
    switch_role(ROLE_EXTRACT)
    prompt = (
        '### Instruction:\n'
        'Extract the key medical facts from this conversation as a single '
        'comma-separated list. Include: chief complaint, duration, severity, '
        'associated symptoms, negated symptoms, relevant history.\n\n'
        f'Conversation:\n{conversation}\n\n'
        '### Response:\n'
        'Key clinical facts: '
    )
    return generate(prompt, EXTRACTION_MAX_TOKENS).split('\n')[0].strip()


DISCLAIMER = (
    '\n\n---\n'
    '⚠️ **Disclaimer:** This is an AI-generated assessment for informational '
    'purposes only. It does NOT constitute medical advice, diagnosis, or treatment. '
    'Always consult a licensed healthcare professional.'
)

def generate_diagnosis(conversation: str, symptoms: str, rag_text: str) -> str:
    """Generates structured diagnostic report using diagnostic_adapter."""
    switch_role(ROLE_DIAGNOSE)

    # Truncate conversation to stay within 2048 token budget
    # Budget: 2048 - 512 (output) - 250 (template) - 500 (RAG) ≈ 786 for conv
    conv = truncate_to_tokens(conversation, MAX_CONV_TOKENS)

    prompt = (
        '### Instruction:\n'
        'You are an experienced medical diagnostic assistant. '
        'Based on the consultation and medical literature below, '
        'provide a structured clinical assessment.\n\n'
        '--- Patient Consultation ---\n'
        f'{conv}\n\n'
        '--- Clinical Summary ---\n'
        f'{symptoms}\n\n'
        '--- Relevant Medical Literature ---\n'
        f'{rag_text}\n\n'
        'Structure your response as:\n'
        '1. Most likely condition: [name] — [reason]\n'
        '2. Other conditions to consider: [name] — [why less likely]\n'
        '3. Recommended next steps: [tests / referral / urgency]\n\n'
        '### Response:\n'
    )
    return generate(prompt, DIAGNOSIS_MAX_TOKENS) + DISCLAIMER


print('✅ Generation functions ready.')

## 🔗 Step 7: Pipeline Orchestrator

In [ ]:
def run_pipeline(session: 'Session') -> str:
    """
    Runs the full post-conversation pipeline:
      1. Extract symptoms  (base model, no adapter)
      2. Retrieve docs     (PubMed + MedlinePlus)
      3. Generate report   (diagnostic_adapter)
      4. Save session      (Drive JSON)

    Returns the final diagnostic report string.
    """
    transcript = session.transcript()

    print('\n Stage 1/3 — Extracting symptoms...')
    session.state   = State.EXTRACTING
    symptoms        = extract_symptoms(transcript)
    session.symptoms = symptoms
    print(f'   ✅ {symptoms}')

    print('\n Stage 2/3 — Retrieving medical literature...')
    session.state   = State.RAG
    docs            = rag.retrieve(symptoms)
    session.rag_docs = docs
    rag_text        = MedicalRAG.format(docs)

    print('\n⚙️  Stage 3/3 — Generating diagnosis...')
    session.state   = State.DIAGNOSING
    diagnosis       = generate_diagnosis(transcript, symptoms, rag_text)
    session.diagnosis = diagnosis
    session.state   = State.DONE

    session.save()
    return diagnosis

print('✅ run_pipeline() defined.')

## 🧪 Step 8: CLI Test (verify pipeline before launching UI)

In [ ]:
# Run this to confirm the full pipeline works end-to-end.
# Uses 3 simulated patient answers so it completes quickly.

session = Session(max_turns=3)

COMPLAINT = 'I have been having severe chest pain for 3 days that radiates to my left arm.'
print(f'👤 Patient: {COMPLAINT}')
session.patient_says(COMPLAINT)

ANSWERS = [
    'It feels like heavy pressure. Gets worse when I climb stairs.',
    'I also feel short of breath and have been sweating more than usual.',
    'I am 52, have high blood pressure, and take lisinopril.',
]

for i, answer in enumerate(ANSWERS, 1):
    if session.exhausted():
        break
    q = ask_followup(session.complaint, session.history_tuples())
    print(f'\n🩺 Doctor (Q{i}): {q}')
    session.doctor_says(q)
    print(f'👤 Patient: {answer}')
    session.patient_says(answer)

print('\n' + '='*65)
print(' Running pipeline...')
print('='*65)
report = run_pipeline(session)

print('\n' + '='*65)
print(' DIAGNOSTIC REPORT')
print('='*65)
print(report)

## 🖥️ Step 9: Gradio Chat UI

In [ ]:
import gradio as gr

_session = None   # one active session per process

WELCOME = (
    'Hello! I am your medical intake assistant. '
    'Please describe your main symptom or health concern, '
    'and I will ask a few follow-up questions before providing an assessment.\n\n'
    '*This assistant does not replace a licensed doctor.*'
)


def respond(user_msg: str, history: list, force: bool = False):
    global _session
    user_msg = (user_msg or '').strip()
    if not user_msg and not force:
        yield history, ''
        return

    if _session is None:
        _session = Session()

    s = _session

    if s.state == State.DONE:
        yield history, '✅ Session complete. Click **Reset** to start a new one.'
        return

    if user_msg:
        s.patient_says(user_msg)
        history = history + [(user_msg, None)]
        yield history, f'⏳ Turn {s.turns + 1}/{s.max_turns}…'

    go_pipeline = force or s.exhausted()

    if not go_pipeline and s.state == State.FOLLOW_UP:
        question = ask_followup(s.complaint, s.history_tuples())
        s.doctor_says(question)
        suffix = ''
        if s.exhausted():
            suffix = '\n\n*(Click **Generate Diagnosis** or reply to trigger the assessment.)*'
        history[-1] = (user_msg, question + suffix)
        yield history, f'Turn {s.turns}/{s.max_turns} complete'

    else:
        history = history + [(None, '🔬 *Analysing your information and retrieving medical literature…*')]
        yield history, '🔬 Running diagnosis pipeline…'
        try:
            report = run_pipeline(s)
        except Exception as e:
            report = f'❌ Error: {e}\n\nPlease reset and try again.'
        history[-1] = (None, report)
        yield history, '✅ Assessment complete. Click **Reset** to start a new session.'


def force_diagnose(history):
    yield from respond('', history, force=True)


def reset():
    global _session
    _session = None
    return [], '', '🆕 New session started.'


with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue   ='blue',
        secondary_hue ='cyan',
        font          =[gr.themes.GoogleFont('Inter'), 'sans-serif'],
    ),
    title='🩺 Medical Agent',
) as demo:

    gr.Markdown(
        '''
        # 🩺 Medical Agent
        AI-powered medical intake assistant. Describe your symptoms, answer a few
        follow-up questions, and receive a preliminary assessment backed by
        PubMed and MedlinePlus literature.

        > ⚠️ This tool does **not** replace a licensed physician.
        '''
    )

    with gr.Row():
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(
                value            =[(None, WELCOME)],
                label            ='Consultation',
                height           =540,
                bubble_full_width=False,
            )
            with gr.Row():
                msg = gr.Textbox(
                    placeholder='Describe your symptoms here…',
                    label='', scale=5, container=False, autofocus=True,
                )
                send = gr.Button('Send ➤', variant='primary', scale=1, min_width=90)

        with gr.Column(scale=1, min_width=200):
            status = gr.Textbox(
                label='Status', value='Waiting for input…',
                lines=3, interactive=False,
            )
            diag_btn  = gr.Button('🔬 Generate Diagnosis Now', variant='secondary', size='sm')
            reset_btn = gr.Button('🔄 Reset Session',          variant='stop',      size='sm')
            gr.Markdown(
                '''
                ---
                **How it works:**
                1. Describe your symptoms
                2. Answer follow-up questions
                3. Agent diagnoses after 6 turns,
                   or click **Generate Diagnosis** any time

                ---
                **Inside the pipeline:**
                - Follow-up questions → `follow_up_adapter`
                - Symptom extraction → base LLaMA-2
                - Literature → PubMed + MedlinePlus
                - Diagnosis → `diagnostic_adapter`
                '''
            )

    send.click(fn=respond, inputs=[msg, chatbot], outputs=[chatbot, status]
               ).then(lambda: '', outputs=msg)
    msg.submit(fn=respond, inputs=[msg, chatbot], outputs=[chatbot, status]
               ).then(lambda: '', outputs=msg)
    diag_btn.click(fn=force_diagnose, inputs=[chatbot], outputs=[chatbot, status])
    reset_btn.click(fn=reset, outputs=[chatbot, msg, status])

print('✅ Gradio UI built. Run Step 10 to launch.')

## 🚀 Step 10: Launch

In [ ]:
demo.launch(
    share      =True,    # generates public URL valid 72h (required on Colab)
    debug      =False,
    show_error =True,
    server_name='0.0.0.0',
    server_port=7860,
)